In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("glass.csv")
x = df.drop("Type", axis = 1).values
y = (df["Type"]==1).astype(int)


In [2]:
x = (x - x.mean(axis=0)) / x.std(axis=0)

In [3]:
np.random.seed(42)
indices = np.random.permutation(len(x))
split = int(0.7 * len(x))
train_idx, test_idx = indices[:split], indices[split:]
x_train, x_test = x[train_idx], x[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [4]:
class Perceptron:

    def __init__(self, lr=0.1, epochs=10000):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.lossTable = []

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def add_bias(self, X):
        bias = np.ones((X.shape[0], 1))
        return np.hstack((X, bias))

    def initialize_weights(self, n_features):
        self.w = np.zeros(n_features + 1)

    def feedForward(self, X):
        z = np.dot(X, self.w)
        h = self.sigmoid(z)
        return h

    def error(self, y, h):
        loss = -np.mean(y * np.log(h + 1e-8) + (1 - y) * np.log(1 - h + 1e-8))
        return loss

    def fit(self, X, y):
        X = self.add_bias(X)
        n_samples, n_features = X.shape
        self.initialize_weights(n_features - 1)

        for epoch in range(self.epochs):
            y_hat = self.feedForward(X)
            gradient = (np.dot(X.T, (y_hat - y)) / n_samples)
            self.w -= self.lr * gradient
            loss = self.error(y, y_hat)
            self.lossTable.append(loss)
            if epoch % (self.epochs // 10) == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")

    def predict_proba(self, X):
        X = self.add_bias(X)
        return self.feedForward(X)

    def predict(self, X, threshold=0.5):
        proba = self.predict_proba(X)
        return (proba >= threshold).astype(int)


In [5]:
model = Perceptron()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)

Epoch 0, Loss: 0.6931
Epoch 1000, Loss: 0.4467
Epoch 2000, Loss: 0.4427
Epoch 3000, Loss: 0.4411
Epoch 4000, Loss: 0.4401
Epoch 5000, Loss: 0.4394
Epoch 6000, Loss: 0.4389
Epoch 7000, Loss: 0.4384
Epoch 8000, Loss: 0.4379
Epoch 9000, Loss: 0.4375


In [6]:
accuracy = np.mean(y_pred == y_test)
print(accuracy)

0.8
